In [14]:
# ====================== THÔNG SỐ DỮ LIỆU ======================
import pandas as pd
import plotly.express as px

# Load dữ liệu (đúng đường dẫn data/raw/, đúng biến theo tên file)
df_nam = pd.read_csv("../data/raw/du_lieu_du_lich_nam.csv")
df_dp  = pd.read_csv("../data/raw/du_lieu_du_lich_dia_phuong.csv")

# Xử lý cột Năm (tách "Sơ bộ 2024" -> 2024, đánh dấu Trạng thái riêng)
df_nam["Trạng thái"] = df_nam["Năm"].astype(str).apply(lambda x: "Sơ bộ" if "Sơ bộ" in x else "Chính thức")
df_nam["Năm"] = df_nam["Năm"].astype(str).str.extract(r"(\d{4})").astype(int)
df_dp["Năm"] = df_dp["Năm"].astype(int)

# Tách cấp dữ liệu (tránh double counting) — chỉ dùng để kiểm tra, không dùng để cộng dồn
df_vung = df_dp[df_dp["Cấp dữ liệu"] == "Vùng"].copy()
df_tinh = df_dp[df_dp["Cấp dữ liệu"] == "Tỉnh/thành"].copy()

print("=" * 60)
print("THÔNG SỐ DỮ LIỆU SỬ DỤNG CHO PHẦN SO SÁNH & XU HƯỚNG")
print("=" * 60)
print(f"• Dataset theo năm          : {df_nam.shape[0]} dòng × {df_nam.shape[1]} cột")
print(f"• Dataset địa phương        : {df_dp.shape[0]} dòng × {df_dp.shape[1]} cột")
print(f"• Số vùng                   : {df_vung['Địa phương'].nunique()} vùng")
print(f"• Số tỉnh/thành phố         : {df_tinh['Địa phương'].nunique()} tỉnh/thành")
print(f"• Giai đoạn phân tích       : 2015 - 2024 (năm 2024 là số liệu sơ bộ)")
print(f"• Đơn vị doanh thu          : Tỷ đồng")
print(f"• Đơn vị lượng khách        : Nghìn lượt")
print("=" * 60)


THÔNG SỐ DỮ LIỆU SỬ DỤNG CHO PHẦN SO SÁNH & XU HƯỚNG
• Dataset theo năm          : 10 dòng × 13 cột
• Dataset địa phương        : 700 dòng × 5 cột
• Số vùng                   : 6 vùng
• Số tỉnh/thành phố         : 63 tỉnh/thành
• Giai đoạn phân tích       : 2015 - 2024 (năm 2024 là số liệu sơ bộ)
• Đơn vị doanh thu          : Tỷ đồng
• Đơn vị lượng khách        : Nghìn lượt


# Tổng quan các chỉ số then chốt

*(Theo dữ liệu thực tế giai đoạn 2015 - 2024)*

| Chỉ tiêu | 2015 | 2019 (đỉnh) | 2021 (đáy) | 2024 (sơ bộ) |
|---|---|---|---|---|
| Doanh thu lưu trú (tỷ đồng) | 44.712 | 67.019 | 23.690 | 94.101 |
| Doanh thu lữ hành (tỷ đồng) | 30.444 | 44.670 | 8.999 | 78.990 |
| Khách lưu trú (nghìn lượt) | 114.011 | 179.365 | 63.603 | 251.897 |
| Khách lữ hành (nghìn lượt) | 12.602 | 18.366 | 3.565 | 34.371 |

> **Lưu ý:** Năm 2024 là số liệu sơ bộ.

# 3.4 Xu hướng Doanh thu & Lượng khách lưu trú (2015 - 2024)

**Câu hỏi:**
Doanh thu lưu trú và lượng khách lưu trú của Việt Nam biến động như thế nào trong giai đoạn 2015 - 2024? Mức độ ảnh hưởng của COVID-19 và tốc độ phục hồi ra sao?.

In [15]:
import pandas as pd
import plotly.express as px
import re

df_nam = pd.read_csv("../data/raw/du_lieu_du_lich_nam.csv")

# Trích số năm ra khỏi chuỗi (vd "Sơ bộ 2024" -> 2024), áp dụng chung cho mọi dòng
df_nam["Năm"] = df_nam["Năm"].astype(str).str.extract(r"(\d{4})").astype(int)

print(df_nam["Năm"].tolist())   # xác nhận giờ đã là số nguyên sạch: 2015...2024

[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


In [16]:
import pandas as pd
import plotly.express as px

df_nam = pd.read_csv("../data/raw/du_lieu_du_lich_nam.csv")
df_nam["Trạng thái"] = df_nam["Năm"].astype(str).apply(lambda x: "Sơ bộ" if "Sơ bộ" in x else "Chính thức")
df_nam["Năm"] = df_nam["Năm"].astype(str).str.extract(r"(\d{4})").astype(int)

df_long = df_nam.melt(
    id_vars=["Năm"],
    value_vars=["Doanh thu cơ sở lưu trú (Tỷ đồng)", "Doanh thu cơ sở lữ hành (Tỷ đồng)"],
    var_name="Chỉ tiêu", value_name="Doanh thu (Tỷ đồng)"
)

fig4 = px.line(
    df_long, x="Năm", y="Doanh thu (Tỷ đồng)", color="Chỉ tiêu", markers=True,
    title="<b>BIỂU ĐỒ XU HƯỚNG DOANH THU CƠ SỞ LƯU TRÚ VÀ CƠ SỞ LỮ HÀNH GIAI ĐOẠN 2015 ĐẾN 2024</b>"
)

fig4.update_layout(
    autosize=False, width=1050, height= 450,
    legend_title_text="",
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="left", x=0),
    hovermode="x unified", plot_bgcolor="white",
    margin=dict(l=60, r=60, t=90, b=100)
)
fig4.update_xaxes(type="linear", tickmode="linear", dtick=1, range=[2014.5, 2024.5], gridcolor="lightgrey")
fig4.update_yaxes(gridcolor="lightgrey", tickformat=",")

fig4.add_vrect(
    x0=2019.5, x1=2021.5, fillcolor="#eb6834", opacity=0.08,
    layer="below", line_width=0,
    annotation_text="Giai đoạn suy giảm", annotation_position="top left",
    annotation_font_size=11, annotation_font_color="grey"
)

fig4.add_annotation(x=2019, y=67019.3, text="Đỉnh trước dịch<br>67.019 tỷ", showarrow=True, arrowhead=1, ay=-50, font=dict(size=11))
fig4.add_annotation(x=2021, y=8998.84, text="Đáy: 8.999 tỷ<br>(-80% so với 2019)", showarrow=True, arrowhead=1, ay=40, font=dict(size=11, color="#d95926"))
fig4.add_annotation(x=2024, y=94101.41, text="2024 (sơ bộ)<br>94.101 tỷ, +40% so đỉnh cũ", showarrow=True, arrowhead=1, ax=-60, ay=-40, font=dict(size=11))

fig4.show()
# Biểu đồ 4
fig4.write_image("../outputs/figures/bieu_do_4_xu_huong_doanh_thu.png", scale=2)
fig4.write_html("../outputs/figures/bieu_do_4_xu_huong_doanh_thu.html")

**Nhận xét:**

Doanh thu lưu trú và lữ hành tăng liên tục 2015-2019, đạt đỉnh **67.019 tỷ** và **44.670 tỷ đồng**. Giai đoạn 2020-2021 giảm mạnh, đáy 2021 còn **23.690 tỷ** (lưu trú, -65%) và **8.999 tỷ** (lữ hành, -80%). Từ 2022 phục hồi nhanh, vượt đỉnh cũ năm 2023; 2024 (sơ bộ) đạt **94.101 tỷ** và **78.990 tỷ đồng**.

**Insight:**
1. Giai đoạn 2020-2021 là điểm gián đoạn lớn nhất trong 10 năm quan sát.

2. Lữ hành giảm sâu hơn lưu trú (80% so với 65%) mức nhạy cảm khác nhau trước cùng một cú sốc.

3. Tốc độ phục hồi (2022-2023) nhanh hơn tốc độ suy giảm, tiếp tục tăng đến 2024.


# 3.5 Xu hướng lượng khách du lịch giai đoạn 2015 - 2024

**Câu hỏi:**
Lượng khách phục vụ tại cơ sở lưu trú và cơ sở lữ hành biến động như thế nào trong giai đoạn 2015 - 2024? Giai đoạn nào giảm mạnh, mức độ phục hồi sau đó ra sao, và kết quả này liên hệ như thế nào với xu hướng doanh thu ở Biểu đồ 4?

Chỉ tiêu sử dụng (đơn vị: Nghìn lượt):

Khách cơ sở lưu trú phục vụ

Khách cơ sở lữ hành phục vụ

In [17]:
import pandas as pd
import plotly.express as px

df_nam = pd.read_csv("../data/raw/du_lieu_du_lich_nam.csv")
df_nam["Trạng thái"] = df_nam["Năm"].astype(str).apply(lambda x: "Sơ bộ" if "Sơ bộ" in x else "Chính thức")
df_nam["Năm"] = df_nam["Năm"].astype(str).str.extract(r"(\d{4})").astype(int)

df_long5 = df_nam.melt(
    id_vars=["Năm"],
    value_vars=["Khách cơ sở lưu trú phục vụ (Nghìn lượt)", "Khách cơ sở lữ hành phục vụ (Nghìn lượt)"],
    var_name="Chỉ tiêu", value_name="Lượng khách (Nghìn lượt)"
)

fig5 = px.line(
    df_long5, x="Năm", y="Lượng khách (Nghìn lượt)", color="Chỉ tiêu", markers=True
)

fig5.update_layout(
    title=dict(
        text="<b>BIỂU ĐỒ XU HƯỚNG KHÁCH DU LỊCH GIAI ĐOẠN 2015 ĐẾN 2024</b>",
        y=0.97, x=0.02, xanchor="left", font=dict(size=18)
    ),
    autosize=False, width=1050, height=600,
    legend_title_text="",
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="left", x=0),
    hovermode="x unified", plot_bgcolor="white",
    margin=dict(l=60, r=40, t=110, b=60)
)
fig5.update_xaxes(type="linear", tickmode="linear", dtick=1, range=[2014.5, 2024.5], gridcolor="lightgrey")
fig5.update_yaxes(gridcolor="lightgrey", tickformat=",")

fig5.add_vrect(
    x0=2019.5, x1=2021.5, fillcolor="#eb6834", opacity=0.08,
    layer="below", line_width=0,
    annotation_text="Giai đoạn suy giảm", annotation_position="top left",
    annotation_font_size=11, annotation_font_color="grey"
)

fig5.add_annotation(x=2019, y=179365.49, text="Đỉnh trước dịch<br>179.365 nghìn lượt", showarrow=True, arrowhead=1, ay=-50, font=dict(size=11))
fig5.add_annotation(x=2021, y=3565.28, text="Đáy: 3.565 nghìn lượt<br>(-81% so với 2019)", showarrow=True, arrowhead=1, ay=40, font=dict(size=11, color="#d95926"))
fig5.add_annotation(x=2024, y=251896.82, text="2024 (sơ bộ)<br>251.897 nghìn lượt, +40% so đỉnh cũ", showarrow=True, arrowhead=1, ax=-60, ay=-40, font=dict(size=11))

fig5.show()
# Biểu đồ 5
fig5.write_image("../outputs/figures/bieu_do_5_xu_huong_luong_khach.png", scale=2)
fig5.write_html("../outputs/figures/bieu_do_5_xu_huong_luong_khach.html")

**Nhận xét:**

Lượng khách lưu trú và lữ hành tăng liên tục 2015-2019, đạt đỉnh **179.365** và **18.366 nghìn lượt**. Giai đoạn 2020-2021 giảm mạnh, đáy 2021 còn **63.603 nghìn lượt** (lưu trú, -65%) và **3.565 nghìn lượt** (lữ hành, -81%). Từ 2022 phục hồi nhanh, vượt đỉnh cũ năm 2023; 2024 (sơ bộ) đạt **251.897** và **34.371 nghìn lượt** cao nhất giai đoạn.

**Insight:**
1. Biến động lượng khách trùng thời điểm với doanh thu ở Biểu đồ 4 (đáy 2021, phục hồi từ 2022), sụt giảm doanh thu gắn liền với sụt giảm quy mô khách.

2. Lữ hành giảm sâu hơn lưu trú (81% so với 65%) cùng mức chênh lệch như ở chỉ tiêu doanh thu.

3. 2019 đến 2024, khách lữ hành tăng nhanh hơn doanh thu lữ hành (+87% so với +77%); lưu trú tăng đồng tốc (+40%), tăng trưởng lữ hành gần đây nghiêng về mở rộng quy mô khách.


# 3.6: Top 10 tỉnh/thành doanh thu du lịch lữ hành cao nhất (2023)
**Câu hỏi:**
Những tỉnh/thành nào dẫn đầu về doanh thu du lịch lữ hành trong năm 2023, và mức độ chênh lệch giữa nhóm dẫn đầu với phần còn lại lớn đến đâu?

In [18]:
df_dp["Doanh thu du lịch lữ hành (Tỷ đồng)"] = pd.to_numeric(
    df_dp["Doanh thu du lịch lữ hành (Tỷ đồng)"], errors="coerce"
)

top10 = df_dp[
    (df_dp["Cấp dữ liệu"] == "Tỉnh/thành") &
    (df_dp["Năm"] == 2023)
].sort_values("Doanh thu du lịch lữ hành (Tỷ đồng)", ascending=False).head(10)

print(top10[["Địa phương", "Doanh thu du lịch lữ hành (Tỷ đồng)"]])

          Địa phương  Doanh thu du lịch lữ hành (Tỷ đồng)
558  TP. Hồ Chí Minh                             25580.00
28            Hà Nội                             20687.30
358          Đà Nẵng                              4579.40
408        Khánh Hòa                              3252.88
58        Quảng Ninh                              1045.97
648       Kiên Giang                               747.43
78         Hải Phòng                               723.83
388        Bình Định                               682.54
108           Hà Nam                               588.10
658          Cần Thơ                               585.40


In [19]:
top10_sorted = top10.sort_values("Doanh thu du lịch lữ hành (Tỷ đồng)")

fig6 = px.bar(
    top10_sorted,
    x="Doanh thu du lịch lữ hành (Tỷ đồng)",
    y="Địa phương",
    orientation="h",
    color="Địa phương",
    text="Doanh thu du lịch lữ hành (Tỷ đồng)",
    color_discrete_sequence=px.colors.qualitative.Set3
)

fig6.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside",
    textfont=dict(size=13),
    cliponaxis=False          # cho phép số hiện ra ngoài vùng vẽ, không bị cắt
)

fig6.update_layout(
    title=dict(
        text="<b>BIỂU ĐỒ TOP 10 TỈNH/THÀNH CÓ DOANH THU DU LỊCH LỮ HÀNH CAO NHẤT (2023)</b>",
        y=0.98, x=0.02, xanchor="left", font=dict(size=18)
    ),
    autosize=False,
    width=1150,                # tăng chiều rộng để có chỗ cho số ở cuối thanh
    height=650,                # tăng chiều cao để 10 thanh không bị chật
    plot_bgcolor="white",
    margin=dict(l=160, r=100, t=100, b=70),   # tăng lề phải (r) để số không bị cắt, lề trái (l) cho tên tỉnh dài
    xaxis_title="Doanh thu (Tỷ đồng)",
    yaxis_title="",
    showlegend=False,
    font=dict(size=13)
)

fig6.update_xaxes(
    type="log",
    gridcolor="lightgrey",
    tickfont=dict(size=11)
)
fig6.update_yaxes(
    tickfont=dict(size=13)
)

fig6.show()
# Biểu đồ 6
fig6.write_image("../outputs/figures/bieu_do_6_top10_dia_phuong.png", scale=2)
fig6.write_html("../outputs/figures/bieu_do_6_top10_dia_phuong.html")

**Nhận xét:**

TP. Hồ Chí Minh và Hà Nội dẫn đầu với **25.580** và **20.687 tỷ đồng**, vượt xa Đà Nẵng (**4.579 tỷ**, hạng 3). Từ hạng 5 trở đi, doanh thu dưới **1.100 tỷ đồng**, không còn chênh lệch lớn; nhóm cuối (Bình Định, Hà Nam, Cần Thơ) quanh **580-680 tỷ đồng**.

**Insight:**
1. Doanh thu tập trung ở 2 đô thị lớn: tổng TP.HCM và Hà Nội (**46.267 tỷ đồng**) vượt tổng 8 địa phương còn lại (**12.206 tỷ đồng**).

2. Cơ cấu phân tầng rõ: nhóm dẫn đầu (>20.000 tỷ), nhóm giữa (>3.000 tỷ), nhóm còn lại (<1.100 tỷ).

3. Các địa phương còn lại ngoài 2 đô thị lớn đều gắn với du lịch biển/trọng điểm vùng (Đà Nẵng, Khánh Hòa, Quảng Ninh, Kiên Giang, Hải Phòng).
